In [15]:

# IMPORTS
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# LOAD DATA
df = pd.read_csv("Software_feature_shipment_data_set.csv")

#  PREPROCESSING
df['planned_shipment_date'] = pd.to_datetime(df['planned_shipment_date'])
df = df.sort_values(by='planned_shipment_date')

# Time features
df['month'] = df['planned_shipment_date'].dt.month
df['day_of_week'] = df['planned_shipment_date'].dt.dayofweek

# Feature engineering
df['complexity_per_dev'] = df['feature_complexity'] / df['team_size']
df['blocker_impact'] = df['num_blockers'] * df['feature_complexity']
df['dependency_pressure'] = df['num_dependencies'] * df['feature_complexity']

# Lag features
df['prev_delay_1'] = df['delay_days'].shift(1)
df['prev_delay_2'] = df['delay_days'].shift(2)

# Rolling mean
df['rolling_delay_mean'] = df['delay_days'].rolling(3).mean()

# Fill missing
df.fillna(0, inplace=True)

# Remove outliers
df = df[df['delay_days'] < df['delay_days'].quantile(0.99)]


# 4. FEATURES
X = df.drop(['delay_days', 'planned_shipment_date'], axis=1)
y = df['delay_days']

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


# BASELINE
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

print("Baseline RF MAE:", round(mean_absolute_error(y_test, rf.predict(X_test)), 4))


# XGBOOST
xgb = XGBRegressor(
    n_estimators=350,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    colsample_bylevel=0.8,
    reg_alpha=2,
    reg_lambda=4,
    random_state=42,
    eval_metric="mae"
)

xgb.fit(X_train, y_train)

print("XGBoost MAE:", round(mean_absolute_error(y_test, xgb.predict(X_test)), 4))


# TUNING
param_grid = {
    "n_estimators": [300, 350],
    "max_depth": [3],
    "learning_rate": [0.04, 0.05],
}

tscv = TimeSeriesSplit(n_splits=3)

grid = GridSearchCV(
    XGBRegressor(
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=2,
        reg_lambda=4,
        random_state=42,
        eval_metric="mae"
    ),
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_xgb = grid.best_estimator_

print("Tuned XGBoost MAE:", round(mean_absolute_error(y_test, best_xgb.predict(X_test)), 4))


#  FINAL PERFORMANCE
print("\nFinal Model Performance:")
print("Train MAE:", round(mean_absolute_error(y_train, best_xgb.predict(X_train)), 4))
print("Test MAE:", round(mean_absolute_error(y_test, best_xgb.predict(X_test)), 4))


# SAFE PREDICTION FUNCTION
def prepare_input(new_input, X_columns):
    df_input = pd.DataFrame([new_input])

    # Add missing columns automatically
    for col in X_columns:
        if col not in df_input.columns:
            df_input[col] = 0

    return df_input[X_columns]


# PREDICTION
planned_date = pd.to_datetime("2026-05-10")

recent_avg_delay = df['delay_days'].tail(3).mean()
smoothed_delay = recent_avg_delay * 0.4

new_input = {
    'team_size': 15,
    'feature_complexity': 6.5,
    'num_dependencies': 2,
    'sprint_length_weeks': 2,
    'num_blockers': 1,
    'holidays_in_sprint': 0,
    'priority_encoded': 1,
    'past_avg_delay_days': 3,
}

# Feature engineering
new_input['month'] = planned_date.month
new_input['day_of_week'] = planned_date.dayofweek
new_input['complexity_per_dev'] = new_input['feature_complexity'] / new_input['team_size']
new_input['blocker_impact'] = new_input['num_blockers'] * new_input['feature_complexity']
new_input['dependency_pressure'] = new_input['num_dependencies'] * new_input['feature_complexity']

# Lag features
new_input['prev_delay_1'] = min(smoothed_delay, 5)
new_input['prev_delay_2'] = min(smoothed_delay * 0.8, 5)
new_input['rolling_delay_mean'] = smoothed_delay

# Prepare safely
input_df = prepare_input(new_input, X.columns)

predicted_delay = best_xgb.predict(input_df)[0]
predicted_delay_days = int(round(predicted_delay))

final_date = planned_date + pd.Timedelta(days=predicted_delay_days)

print("\nNext Shipment Prediction:")
print("Planned Date:", planned_date.date())
print("Predicted Delay:", predicted_delay_days, "days")
print("Final Shipment Date:", final_date.date())


#  EXPLANATION
def explain(features, delay):
    reasons = []

    if features['feature_complexity'] > 6:
        reasons.append("high complexity")
    if features['num_blockers'] > 0:
        reasons.append("blockers")
    if features['num_dependencies'] > 1:
        reasons.append("dependencies")
    if features['prev_delay_1'] > 4:
        reasons.append("recent delays")

    return f"Delay of {delay} days due to " + ", ".join(reasons)


print("\nExplanation:")
print(explain(new_input, predicted_delay_days))

Baseline RF MAE: 1.0772
XGBoost MAE: 0.9135
Tuned XGBoost MAE: 0.8998

Final Model Performance:
Train MAE: 0.5808
Test MAE: 0.8998

Next Shipment Prediction:
Planned Date: 2026-05-10
Predicted Delay: 15 days
Final Shipment Date: 2026-05-25

Explanation:
Delay of 15 days due to high complexity, blockers, dependencies, recent delays


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("\nMODEL METRICS")

test_preds = best_xgb.predict(X_test)

mae = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2 = r2_score(y_test, test_preds)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))

# Accuracy-like metric
within_2_days = np.mean(np.abs(y_test - test_preds) <= 2)
print("Accuracy (±2 days):", round(within_2_days, 4))


MODEL METRICS
MAE: 0.8998
RMSE: 1.1094
R2 Score: 0.9443
Accuracy (±2 days): 0.9294


In [19]:

# USER INPUT SECTION
print("\nEnter details for next shipment prediction:")

team_size = int(input("Team size: "))
feature_complexity = float(input("Feature complexity (1-10): "))
num_dependencies = int(input("Number of dependencies: "))
sprint_length = int(input("Sprint length (weeks): "))
num_blockers = int(input("Number of blockers: "))
holidays = int(input("Holidays in sprint: "))
priority = int(input("Priority (encoded 0/1/2): "))
past_delay = float(input("Past average delay days: "))
planned_date = pd.to_datetime(input("Planned shipment date (YYYY-MM-DD): "))

# BUILD INPUT
recent_avg_delay = df['delay_days'].tail(3).mean()
smoothed_delay = recent_avg_delay * 0.4

new_input = {
    'team_size': team_size,
    'feature_complexity': feature_complexity,
    'num_dependencies': num_dependencies,
    'sprint_length_weeks': sprint_length,
    'num_blockers': num_blockers,
    'holidays_in_sprint': holidays,
    'priority_encoded': priority,
    'past_avg_delay_days': past_delay,
}

# FEATURE ENGINEERING
new_input['month'] = planned_date.month
new_input['day_of_week'] = planned_date.dayofweek
new_input['complexity_per_dev'] = new_input['feature_complexity'] / new_input['team_size']
new_input['blocker_impact'] = new_input['num_blockers'] * new_input['feature_complexity']
new_input['dependency_pressure'] = new_input['num_dependencies'] * new_input['feature_complexity']
# Lag features
new_input['prev_delay_1'] = min(smoothed_delay, 5)
new_input['prev_delay_2'] = min(smoothed_delay * 0.8, 5)
new_input['rolling_delay_mean'] = smoothed_delay
# PREDICTION
input_df = prepare_input(new_input, X.columns)

predicted_delay = best_xgb.predict(input_df)[0]
predicted_delay_days = int(round(predicted_delay))

final_date = planned_date + pd.Timedelta(days=predicted_delay_days)

print("\n PREDICTION RESULT ")
print("Planned Date:", planned_date.date())
print("Predicted Delay:", predicted_delay_days, "days")
print("Final Shipment Date:", final_date.date())

# EXPLANATION
print("\nExplanation:")
print(explain(new_input, predicted_delay_days))


Enter details for next shipment prediction:
Team size: 2
Feature complexity (1-10): 2
Number of dependencies: 2
Sprint length (weeks): 1
Number of blockers: 2
Holidays in sprint: 2
Priority (encoded 0/1/2): 0
Past average delay days: 4
Planned shipment date (YYYY-MM-DD): 2025-12-12

 PREDICTION RESULT 
Planned Date: 2025-12-12
Predicted Delay: 13 days
Final Shipment Date: 2025-12-25

Explanation:
Delay of 13 days due to blockers, dependencies, recent delays


In [18]:

# ACTUAL vs PREDICTED ANALYSIS


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Predict on test set
test_preds = best_xgb.predict(X_test)

# CREATE COMPARISON DATAFRAME

comparison_df = pd.DataFrame({
    "Actual Delay": y_test.values,
    "Predicted Delay": test_preds
})

# ERROR METRICS

comparison_df["Error"] = comparison_df["Actual Delay"] - comparison_df["Predicted Delay"]

print("\n SAMPLE OUTPUT ")
print(comparison_df.head(15))



 SAMPLE OUTPUT 
    Actual Delay  Predicted Delay     Error
0              9         8.757755  0.242245
1             14        13.451964  0.548036
2              7         7.399239 -0.399239
3             15        15.450785 -0.450785
4              9         8.509271  0.490729
5             23        20.467190  2.532810
6             13         9.437246  3.562754
7             16        16.191267 -0.191267
8             11        13.023446 -2.023446
9             13        14.252634 -1.252634
10            12        12.232469 -0.232469
11             5         6.048013 -1.048013
12            14        13.429491  0.570509
13             9         7.690859  1.309141
14            21        21.383085 -0.383085
